In [ ]:
import random
from datetime import datetime, timedelta
from collections import defaultdict
from pathlib import Path
from itertools import cycle
import pandas as pd

random.seed(42)

# -----------------------------
# 1) 读取维表
# -----------------------------
geo_df = pd.read_parquet("output/01_geo_dim.parquet")
url_dim_df = pd.read_parquet("output/02_url_category_dim.parquet")

print("geo_dim sample:")
print(geo_df.head())

print("url_category_dim sample:")
print(url_dim_df.head())

# 取国家列表
country_codes = geo_df["country_code"].dropna().unique().tolist()
country_weight_map = {
    "CN": 12,    "US": 8,    "IN": 6,    "JP": 4,    "DE": 3,    "GB": 3,    "FR": 3,
    "KR": 2,    "BR": 2,    "AU": 2,    "CA": 2,    "SG": 1,    "AE": 1,    "AR": 1,
    "EG": 1,    "ES": 1,    "ID": 1,    "IT": 1,    "KE": 1,    "MX": 1,    "NG": 1,
    "NL": 1,    "NO": 1,    "NZ": 1,    "RU": 1,    "SA": 1,    "SE": 1,    "TH": 1,
    "TR": 1,    "UA": 1,    "VN": 1,    "ZA": 1,}

# 目录里所有 category 及其 path_pattern
category_path_map = defaultdict(list)
for _, row in url_dim_df[["category", "path_pattern"]].iterrows():
    category_path_map[row["category"]].append(row["path_pattern"])

# -----------------------------
# 2) 生成规则
# -----------------------------
METHODS = ["GET", "POST", "PUT", "DELETE", "PATCH", "HEAD"]
STATUS_DIST = [200, 200, 200, 200, 200, 200, 200, 200, 200, 200, 404, 500, 302, 301]
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
    "Mozilla/5.0 (Linux; Android 14; Pixel 8) AppleWebKit/537.36",
    "Mozilla/5.0 (iPhone; CPU iPhone OS 17_0 like Mac OS X) AppleWebKit/605.1.15",
    "curl/8.0.1",
    "PostmanRuntime/7.36.0",
    "python-requests/2.31.0"
]

HOURLY_WEIGHT = {
    0: 0.3, 1: 0.2, 2: 0.2, 3: 0.2, 4: 0.2, 5: 0.5,
    6: 0.8, 7: 1.2, 8: 1.8, 9: 2.5, 10: 2.8, 11: 3.0,
    12: 3.2, 13: 3.0, 14: 2.8, 15: 2.9, 16: 3.1, 17: 3.4,
    18: 3.2, 19: 2.7, 20: 2.0, 21: 1.5, 22: 1.0, 23: 0.6
}

CATEGORY_WEIGHT = {
    "home": 18,
    "product": 17,
    "category": 14,
    "api": 12,
    "login": 7,
    "user": 8,
    "static": 10,
    "admin": 3,
    "search": 7,
    "checkout": 4,
    "error": 3,
    "other": 7
}

# -----------------------------
# 3) 生成 IP
# -----------------------------
def gen_ip(country_code: str):
    base = sum(ord(c) for c in country_code) % 200
    a = (base + random.randint(1, 90)) % 255
    b = (base * 3 + random.randint(1, 120)) % 255
    c = (base * 5 + random.randint(1, 150)) % 255
    d = random.randint(1, 254)
    return f"{a}.{b}.{c}.{d}"

# -----------------------------
# 4) 生成 URL / path / referer
# -----------------------------
def random_path_by_category(category: str):
    paths = category_path_map.get(category, ["/"])
    p = random.choice(paths)

    if "?" in p:
        path = p.split("?")[0]
        query = p.split("?", 1)[1]
        return f"https://shop.example.com{path}?{query}", path
    return f"https://shop.example.com{p}", p


# 增强真实感的内部/外部 Referer 区分
def generate_referrer(category: str):
    internal_refs = [
        "https://shop.example.com/",
        "https://shop.example.com/product/1001",
        "https://shop.example.com/product/1002",
        "https://shop.example.com/search?q=laptop",
        "https://shop.example.com/search?q=phone",
        "https://shop.example.com/category/electronics",
        "https://shop.example.com/category/home",
        "https://shop.example.com/login",
        "https://shop.example.com/user/profile",
        "https://shop.example.com/user/orders",
        "https://shop.example.com/checkout",
        "https://shop.example.com/"
    ]

    external_refs = [
        "https://www.baidu.com/s?wd=shop",
        "https://www.google.com/search?q=shop+example",
        "https://weibo.com/search?q=shop",
        "https://www.zhihu.com/search?q=shop+example",
        "https://s.taobao.com/search?q=shop",
        "https://www.bing.com/search?q=shop+example"
    ]

    # 让错误页更偏向站内错误路径，增强真实感
    if category == "error" and random.random() < 0.7:
        return random.choice([
            "https://shop.example.com/404",
            "https://shop.example.com/500",
            "https://shop.example.com/unknown"
        ])

    p = random.random()
    if p < 0.70:
        return random.choice(internal_refs)
    elif p < 0.90:
        return None   # 无 Referer / 直接访问
    else:
        return random.choice(external_refs)

# -----------------------------
# 5) 生成日志记录
# -----------------------------
total_rows = 600_000
base_date = datetime(2024, 1, 1)
records = []

# 按日期权重分配：工作日比周末更高，且有自然波动
DAY_WEIGHTS = [0.18, 0.14, 0.13, 0.11, 0.12, 0.10, 0.09, 0.07, 0.04, 0.02]
day_offsets = random.choices(
    population=list(range(10)),
    weights=DAY_WEIGHTS,
    k=total_rows
)
# 国家按真实分布加权
country_codes_weighted = []
country_weights = []
for c in country_codes:
    if c in country_weight_map:
        w = country_weight_map[c]
    else:
        w = 0.5
    country_codes_weighted.append(c)
    country_weights.append(w)

for i, day_offset in enumerate(day_offsets):
    # 10 天时间分布
    current_date = base_date + timedelta(days=day_offset)

    # 按真实小时分布采样
    hour = random.choices(
        list(HOURLY_WEIGHT.keys()),
        weights=[HOURLY_WEIGHT[h] for h in HOURLY_WEIGHT],
        k=1
    )[0]
    minute = random.randint(0, 59)
    second = random.randint(0, 59)

    # 业务分类采样
    category = random.choices(
        list(CATEGORY_WEIGHT.keys()),
        weights=[CATEGORY_WEIGHT[k] for k in CATEGORY_WEIGHT],
        k=1
    )[0]

    # 国家/IP
    country_code = random.choices(
        country_codes_weighted,
        weights=country_weights,
        k=1
    )[0]
    ip = gen_ip(country_code)



    # 异常流量增强
    if random.random() < 0.01:
        ip = random.choice([
            "10.10.10.10", "20.20.20.20", "43.21.12.34",
            "101.18.25.90", "210.55.99.7"
        ])

    if category == "error" and random.random() < 0.8:
        status = random.choice([404, 500, 503])
    else:
        status = random.choice(STATUS_DIST)

    method = random.choice(METHODS)
    if category == "api":
        method = random.choice(["GET", "POST", "PUT", "DELETE"])
    elif category == "static":
        method = "GET"
    elif category == "checkout" and random.random() < 0.7:
        method = "POST"

    url, path = random_path_by_category(category)
    referer = generate_referrer(category)

    # 访问量高峰期更容易慢请求
    if hour in [9, 10, 11, 12, 13, 14, 15, 16, 17, 18] and random.random() < 0.18:
        response_time = random.randint(700, 5000)
    elif status >= 500:
        response_time = random.randint(300, 4000)
    else:
        response_time = random.randint(20, 600)

    # 响应字节数
    if status >= 400:
        bytes_sent = random.randint(120, 9000)
    else:
        bytes_sent = random.randint(200, 150000)

    user_agent = random.choice(USER_AGENTS)

    timestamp = datetime(
        current_date.year, current_date.month, current_date.day,
        hour, minute, second
    )

    log_id = f"{current_date.strftime('%Y%m%d')}-{i:08d}"

    records.append({
        "log_id": log_id,
        "timestamp": timestamp,
        "date": current_date.strftime("%Y-%m-%d"),
        "hour": hour,
        "ip": ip,
        "method": method,
        "url": url,
        "path": path,
        "status_code": status,
        "response_time_ms": response_time,
        "bytes": bytes_sent,
        "user_agent": user_agent,
        "country": country_code,
        "referer": referer,
        "is_anomaly": 0,
        "anomaly_type": None,
    })

    if i % 100000 == 0:
        print(f"已生成 {i} 条日志")


# -----------------------------
# 5.1) 在主表内注入异常流量（推荐）
# -----------------------------
# 目标：主表仍为 600_000 行
# 异常记录在主表内部覆盖/修改，数量约 4,000
# 所有异常路径均来自原始 URL 维表，不额外创造新 path

real_paths = url_dim_df["path_pattern"].dropna().astype(str).tolist()
real_paths = [p for p in real_paths if p.startswith("/")]
real_paths = list(dict.fromkeys(real_paths))

# 先准备异常的目标索引
anomaly_target_count = 4000
anomaly_indices = random.sample(range(total_rows), anomaly_target_count)

# 统一的注入函数：直接修改已有 records
def inject_anomaly_record(record, ip, path, method, status, response_time_ms, bytes_sent, anomaly_type):
    record["ip"] = ip
    record["method"] = method
    record["url"] = f"https://shop.example.com{path}"
    record["path"] = path
    record["status_code"] = status
    record["response_time_ms"] = response_time_ms
    record["bytes"] = bytes_sent
    record["is_anomaly"] = 1
    record["anomaly_type"] = anomaly_type
    return record

# 1) 高频 IP 扫描 / 攻击
# 约 2,000 条
scan_ips = [
    "10.10.10.10", "20.20.20.20", "43.21.12.34", "101.18.25.90",
    "210.55.99.7", "198.51.100.8", "203.0.113.6", "192.0.2.11"
]

scan_idx_pool = random.sample(anomaly_indices, 2000)
for idx, ip in zip(scan_idx_pool, cycle(scan_ips)):
    row = records[idx]
    path = random.choice(real_paths)
    method = random.choice(["GET", "POST", "HEAD", "PUT"])
    status = random.choice([403, 404, 500, 503])
    rt = random.randint(4000, 15000)
    byte_count = random.randint(300, 20000)

    inject_anomaly_record(
        row,
        ip=ip,
        path=path,
        method=method,
        status=status,
        response_time_ms=rt,
        bytes_sent=byte_count,
        anomaly_type="scan_ip"
    )

# 2) 突发流量（Burst）
# 约 1,000 条
burst_idx_pool = random.sample([i for i in anomaly_indices if i not in scan_idx_pool], 1000)
burst_paths = [p for p in real_paths if p.startswith("/search") or p.startswith("/admin") or p.startswith("/checkout")]
for idx in burst_idx_pool:
    row = records[idx]
    ip = random.choice(scan_ips + ["58.99.12.56", "120.52.88.40", "89.12.44.9"])
    path = random.choice(burst_paths)
    method = random.choice(["GET", "POST", "HEAD"])
    status = random.choices([200, 302, 404, 500, 503], weights=[30, 20, 20, 18, 12], k=1)[0]
    rt = random.randint(3000, 12000) if status >= 400 else random.randint(800, 7000)
    byte_count = random.randint(512, 120000)

    inject_anomaly_record(
        row,
        ip=ip,
        path=path,
        method=method,
        status=status,
        response_time_ms=rt,
        bytes_sent=byte_count,
        anomaly_type="burst"
    )

# 3) 高错误率 + 慢请求
# 约 800 条
error_idx_pool = random.sample([i for i in anomaly_indices if i not in scan_idx_pool and i not in burst_idx_pool], 800)
error_paths = [p for p in real_paths if p.startswith("/login") or p.startswith("/admin") or p.startswith("/api") or p.startswith("/checkout")]
for idx in error_idx_pool:
    row = records[idx]
    ip = random.choice(scan_ips + ["114.32.87.8", "61.180.22.15", "5.6.7.8"])
    path = random.choice(error_paths)
    method = random.choice(["GET", "POST", "PUT", "DELETE"])
    status = random.choice([403, 404, 500, 503])
    rt = random.randint(3000, 15000)
    byte_count = random.randint(200, 12000)

    inject_anomaly_record(
        row,
        ip=ip,
        path=path,
        method=method,
        status=status,
        response_time_ms=rt,
        bytes_sent=byte_count,
        anomaly_type="high_error_slow"
    )

# 4) 定向路径攻击
# 约 200 条
target_idx_pool = list(set(anomaly_indices) - set(scan_idx_pool) - set(burst_idx_pool) - set(error_idx_pool))
target_idx_pool = random.sample(target_idx_pool, 200)

target_paths = [p for p in real_paths if p.startswith("/admin") or p.startswith("/login") or p.startswith("/api")]
for idx in target_idx_pool:
    row = records[idx]
    ip = random.choice(scan_ips)
    path = random.choice(target_paths)
    method = random.choice(["GET", "POST", "PUT"])
    status = random.choice([401, 403, 404, 500])
    rt = random.randint(2500, 10000)
    byte_count = random.randint(300, 9000)

    inject_anomaly_record(
        row,
        ip=ip,
        path=path,
        method=method,
        status=status,
        response_time_ms=rt,
        bytes_sent=byte_count,
        anomaly_type="targeted_path_attack"
    )

# 记录总注入数
anomaly_count = sum(1 for r in records if r.get("is_anomaly") == 1)
print(f"主表中异常记录数: {anomaly_count}")


# -----------------------------
# 6) 转 DataFrame 并保存
# -----------------------------
df = pd.DataFrame(records)

# 字段类型整理
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hour"] = df["hour"].astype(int)
df["status_code"] = df["status_code"].astype(int)
df["response_time_ms"] = df["response_time_ms"].astype(int)
df["bytes"] = df["bytes"].astype(int)

# 这个版本先不做分区，保留 date 字段；后续上传到 HDFS 后再按 Spark 的 date 分区写出
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "03_http_logs.parquet"

df.to_parquet(output_path, index=False)

print(f"总记录数: {len(df)}")
print(df.head(10))
print(f"已写出到: {output_path}")

geo_dim sample:
  country_code          country_name         region      continent  \
0           AE  United Arab Emirates    Middle East           Asia   
1           AR             Argentina  South America  South America   
2           AU             Australia        Oceania        Oceania   
3           BR                Brazil  South America  South America   
4           CA                Canada  North America  North America   

   is_high_risk  
0         False  
1          True  
2         False  
3          True  
4         False  
url_category_dim sample:
         path_pattern category  is_api  is_static  priority
0                   /     home   False      False         4
1                /404    error   False      False         1
2                /500    error   False      False         1
3              /about    other   False      False         1
4  /account/favorites     user   False      False         4
已生成 0 条日志
已生成 100000 条日志
已生成 200000 条日志
已生成 300000 条日志
已生成 400000 条日志


In [3]:
debug1_df = pd.read_parquet(r"output\\03_http_logs.parquet")
country_counts = debug1_df["country"].value_counts()
country_counts

country
CN    106075
US     70835
IN     52669
JP     35008
FR     26753
GB     26624
DE     26336
AU     17680
CA     17562
BR     17539
KR     17499
SE      9035
SA      8990
NL      8967
RU      8960
ID      8952
AE      8895
TR      8871
AR      8866
VN      8852
SG      8843
EG      8842
NZ      8822
NG      8779
ES      8758
MX      8757
KE      8744
TH      8742
ZA      8710
NO      8705
UA      8687
IT      8643
Name: count, dtype: int64

In [4]:
referer_counts = debug1_df["referer"].value_counts()
referer_counts

referer
https://shop.example.com/                        68373
https://shop.example.com/search?q=laptop         34539
https://shop.example.com/category/electronics    34469
https://shop.example.com/product/1001            34408
https://shop.example.com/login                   34350
https://shop.example.com/user/orders             34258
https://shop.example.com/search?q=phone          34238
https://shop.example.com/category/home           34212
https://shop.example.com/product/1002            34208
https://shop.example.com/checkout                34185
https://shop.example.com/user/profile            34030
https://www.baidu.com/s?wd=shop                   9977
https://www.google.com/search?q=shop+example      9964
https://s.taobao.com/search?q=shop                9904
https://www.zhihu.com/search?q=shop+example       9881
https://www.bing.com/search?q=shop+example        9858
https://weibo.com/search?q=shop                   9761
https://shop.example.com/404                      3916
ht

In [5]:
import gc
gc.collect()

9